# 2026 Bleaching Dataset Prep

This notebook reads point annotations from `annotations.csv`, downloads source images from the image bucket, extracts centered crops for the three bleaching classes, and uploads the crops plus manifest files to the crops bucket.

Expected classes from the current CSV: `*UNK`, `*CORAL`, `*CORAL_BL`.

In [1]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    'google.cloud.storage': 'google-cloud-storage',
    'PIL': 'pillow',
    'pandas': 'pandas',
    'tqdm': 'tqdm',
}


def module_is_available(module_name: str) -> bool:
    try:
        importlib.import_module(module_name)
        return True
    except ModuleNotFoundError:
        return False


missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if not module_is_available(module_name)
]

if missing_packages:
    try:
        import pip  # noqa: F401
    except ModuleNotFoundError:
        subprocess.run([sys.executable, '-m', 'ensurepip', '--upgrade'], check=True)

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *sorted(set(missing_packages))], check=True)
    print({'installed_packages': sorted(set(missing_packages))})
else:
    print('Required packages already available.')

{'installed_packages': ['google-cloud-storage', 'pandas', 'tqdm']}


In [2]:
from __future__ import annotations

from collections import Counter
from io import BytesIO
from pathlib import Path

import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from google.cloud import storage

/home/conda/envs/coral-train/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ANNOTATIONS_CSV = Path('annotations.csv')
IMAGE_BUCKET_URI = 'gs://nmfs_odp_pifsc/PIFSC/ESD/ARP/pifsc-ai-data-repository/coralnet_mirror/NOAA_ESD_Coral_Bleaching_Classifier/images'
CROPS_BUCKET_URI = 'gs://nmfs_odp_pifsc/PIFSC/ESD/ARP/pifsc-ai-data-repository/coralnet_mirror/NOAA_ESD_Coral_Bleaching_Classifier/crops'

CLASS_MAP = {
    '*UNK': 'UNK',
    '*CORAL': 'CORAL',
    '*CORAL_BL': 'CORAL_BL',
}

CROP_SIZE = 224
JPEG_QUALITY = 95
OVERWRITE_EXISTING = False
MAX_IMAGES = None
STRICT_MISSING_IMAGES = False
CROP_DESTINATION_MODE = 'local'  # 'gcs' uploads crops to CROPS_BUCKET_URI, 'local' saves crops in the cloud workstation training folder

LOCAL_OUTPUT_DIR = Path('/tmp/bleaching_dataset_prep')
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_MANIFEST_PATH = LOCAL_OUTPUT_DIR / 'crop_manifest.csv'
LOCAL_COUNTS_PATH = LOCAL_OUTPUT_DIR / 'class_counts.csv'

LOCAL_TRAINING_ROOT = Path.home() / 'bleaching_classifier_training'
LOCAL_SOURCE_IMAGES_DIR = LOCAL_TRAINING_ROOT / 'source_images'
LOCAL_CROPS_DIR = LOCAL_TRAINING_ROOT / 'crops'
LOCAL_MANIFEST_DIR = LOCAL_CROPS_DIR / '_manifests'

LOCAL_TRAINING_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_SOURCE_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CROPS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

if CROP_DESTINATION_MODE not in {'gcs', 'local'}:
    raise ValueError(f'Unsupported CROP_DESTINATION_MODE: {CROP_DESTINATION_MODE}')

print(f'Annotations: {ANNOTATIONS_CSV.resolve()}')
print(f'Local cache: {LOCAL_OUTPUT_DIR}')
print({'crop_destination_mode': CROP_DESTINATION_MODE, 'local_training_root': str(LOCAL_TRAINING_ROOT)})

Annotations: /home/user/v2/annotations.csv
Local cache: /tmp/bleaching_dataset_prep
{'crop_destination_mode': 'local', 'local_training_root': '/home/user/bleaching_classifier_training'}


In [4]:
def parse_gs_uri(uri: str) -> tuple[str, str]:
    if not uri.startswith('gs://'):
        raise ValueError(f'Expected a gs:// URI, got: {uri}')
    bucket_name, _, prefix = uri[5:].partition('/')
    return bucket_name, prefix.strip('/')


def build_blob_path(prefix: str, name: str) -> str:
    clean_name = name.lstrip('/')
    return f'{prefix}/{clean_name}' if prefix else clean_name


def centered_crop_box(width: int, height: int, center_x: int, center_y: int, crop_size: int) -> tuple[int, int, int, int]:
    crop_width = min(crop_size, width)
    crop_height = min(crop_size, height)

    left = max(center_x - crop_width // 2, 0)
    top = max(center_y - crop_height // 2, 0)

    max_left = max(width - crop_width, 0)
    max_top = max(height - crop_height, 0)
    left = min(left, max_left)
    top = min(top, max_top)

    right = left + crop_width
    bottom = top + crop_height
    return left, top, right, bottom


def make_crop_name(image_name: str, annotation_id: int, row: int, column: int) -> str:
    stem = Path(image_name).stem.replace(' ', '_')
    return f'{stem}__ann{annotation_id:06d}__r{row}__c{column}.jpg'

In [5]:
annotations = pd.read_csv(ANNOTATIONS_CSV)
annotations = annotations.rename(columns={'Name': 'image_name', 'Row': 'row', 'Column': 'column', 'Label code': 'label_code'})
annotations = annotations[['image_name', 'row', 'column', 'label_code']].copy()
annotations['row'] = annotations['row'].round().astype(int)
annotations['column'] = annotations['column'].round().astype(int)
annotations = annotations[annotations['label_code'].isin(CLASS_MAP)].copy()
annotations['class_name'] = annotations['label_code'].map(CLASS_MAP)
annotations = annotations.reset_index(names='annotation_id')

class_counts = annotations['class_name'].value_counts().rename_axis('class_name').reset_index(name='annotation_count')
display(class_counts)
print(f'Total annotations retained: {len(annotations):,}')
print(f'Unique images retained: {annotations["image_name"].nunique():,}')

/tmp/ipykernel_1916/3145723230.py:1: DtypeWarning: Columns (0: Analyst) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations = pd.read_csv(ANNOTATIONS_CSV)


,class_name,annotation_count
0,UNK,37638
1,CORAL,8028
2,CORAL_BL,4912


Total annotations retained: 50,578
Unique images retained: 6,140


In [6]:
image_bucket_name, image_prefix = parse_gs_uri(IMAGE_BUCKET_URI)
crops_bucket_name, crops_prefix = parse_gs_uri(CROPS_BUCKET_URI)

storage_client = storage.Client()
image_bucket = storage_client.bucket(image_bucket_name)
crops_bucket = storage_client.bucket(crops_bucket_name)

print(f'Image bucket: {image_bucket_name}')
print(f'Image prefix: {image_prefix}')
print(f'Crops bucket: {crops_bucket_name}')
print(f'Crops prefix: {crops_prefix}')

# Build a lookup so annotations can find images even when objects are nested under subfolders.
image_blob_lookup = {}
duplicate_image_names = set()
for blob in tqdm(storage_client.list_blobs(image_bucket_name, prefix=image_prefix), desc='Indexing source images'):
    image_name = Path(blob.name).name
    if image_name in image_blob_lookup:
        duplicate_image_names.add(image_name)
        continue
    image_blob_lookup[image_name] = blob.name

print({
    'indexed_unique_image_names': len(image_blob_lookup),
    'duplicate_base_names_skipped': len(duplicate_image_names),
})

Image bucket: nmfs_odp_pifsc
Image prefix: PIFSC/ESD/ARP/pifsc-ai-data-repository/coralnet_mirror/NOAA_ESD_Coral_Bleaching_Classifier/images
Crops bucket: nmfs_odp_pifsc
Crops prefix: PIFSC/ESD/ARP/pifsc-ai-data-repository/coralnet_mirror/NOAA_ESD_Coral_Bleaching_Classifier/crops


Indexing source images: 5218it [00:00, 9315.59it/s] 

{'indexed_unique_image_names': 5218, 'duplicate_base_names_skipped': 0}


In [7]:
# Build a lookup so image names can be resolved even when bucket objects are nested in subfolders.
image_name_to_paths = {}
for blob in tqdm(storage_client.list_blobs(image_bucket_name, prefix=image_prefix), desc='Indexing image bucket'):
    if blob.name.endswith('/') or blob.name.endswith('_SUCCESS'):
        continue
    basename = Path(blob.name).name
    image_name_to_paths.setdefault(basename, []).append(blob.name)

sample_image = annotations.iloc[0]['image_name']
sample_blob_path = build_blob_path(image_prefix, sample_image)
sample_exists = image_bucket.blob(sample_blob_path).exists(storage_client)

if sample_exists:
    resolved_sample_blob_path = sample_blob_path
else:
    candidates = image_name_to_paths.get(sample_image, [])
    resolved_sample_blob_path = candidates[0] if len(candidates) == 1 else None

print({
    'sample_image': sample_image,
    'direct_blob_path': sample_blob_path,
    'direct_exists': sample_exists,
    'resolved_blob_path': resolved_sample_blob_path,
    'candidate_count': len(image_name_to_paths.get(sample_image, [])),
})

if resolved_sample_blob_path is None and STRICT_MISSING_IMAGES:
    raise FileNotFoundError(
        'Sample image could not be uniquely resolved in GCS. '
        'Set STRICT_MISSING_IMAGES=False to continue and skip unresolved rows.'
    )

Indexing image bucket: 5218it [00:00, 9231.84it/s]


{'sample_image': 'AGU-523_2022_A_23.JPG', 'direct_blob_path': 'PIFSC/ESD/ARP/pifsc-ai-data-repository/coralnet_mirror/NOAA_ESD_Coral_Bleaching_Classifier/images/AGU-523_2022_A_23.JPG', 'direct_exists': False, 'resolved_blob_path': None, 'candidate_count': 0}


In [13]:
grouped = list(annotations.groupby('image_name', sort=False))
if MAX_IMAGES is not None:
    grouped = grouped[:MAX_IMAGES]

results = []
skipped = Counter()
written_crops = 0
downloaded_source_images = 0
missing_images = set()
ambiguous_images = set()

for image_name, image_rows in tqdm(grouped, desc='Extracting crops'):
    direct_blob_path = build_blob_path(image_prefix, image_name)
    direct_blob = image_bucket.blob(direct_blob_path)

    if direct_blob.exists(storage_client):
        image_blob_path = direct_blob_path
    else:
        candidates = image_name_to_paths.get(image_name, [])
        if len(candidates) == 1:
            image_blob_path = candidates[0]
        elif len(candidates) == 0:
            skipped['missing_source_image'] += len(image_rows)
            missing_images.add(image_name)
            continue
        else:
            skipped['ambiguous_source_image'] += len(image_rows)
            ambiguous_images.add(image_name)
            continue

    image_blob = image_bucket.blob(image_blob_path)
    image_bytes = image_blob.download_as_bytes()
    image = Image.open(BytesIO(image_bytes))
    image = ImageOps.exif_transpose(image).convert('RGB')
    width, height = image.size

    local_source_image_path = None
    if CROP_DESTINATION_MODE == 'local':
        local_source_image_path = LOCAL_SOURCE_IMAGES_DIR / Path(image_name).name
        if OVERWRITE_EXISTING or not local_source_image_path.exists():
            local_source_image_path.write_bytes(image_bytes)
            downloaded_source_images += 1

    for row in image_rows.itertuples(index=False):
        left, top, right, bottom = centered_crop_box(width, height, row.column, row.row, CROP_SIZE)
        crop = image.crop((left, top, right, bottom))

        crop_name = make_crop_name(row.image_name, row.annotation_id, row.row, row.column)

        if CROP_DESTINATION_MODE == 'gcs':
            crop_output_path = build_blob_path(crops_prefix, f'{row.class_name}/{crop_name}')
            crop_blob = crops_bucket.blob(crop_output_path)

            if crop_blob.exists(storage_client) and not OVERWRITE_EXISTING:
                skipped['already_exists'] += 1
            else:
                buffer = BytesIO()
                crop.save(buffer, format='JPEG', quality=JPEG_QUALITY, optimize=True)
                buffer.seek(0)
                crop_blob.upload_from_file(buffer, content_type='image/jpeg')
                written_crops += 1
        else:
            local_class_dir = LOCAL_CROPS_DIR / row.class_name
            local_class_dir.mkdir(parents=True, exist_ok=True)
            local_crop_path = local_class_dir / crop_name
            crop_output_path = str(local_crop_path)

            if local_crop_path.exists() and not OVERWRITE_EXISTING:
                skipped['already_exists'] += 1
            else:
                crop.save(local_crop_path, format='JPEG', quality=JPEG_QUALITY, optimize=True)
                written_crops += 1

        results.append({
            'annotation_id': row.annotation_id,
            'image_name': row.image_name,
            'label_code': row.label_code,
            'class_name': row.class_name,
            'row': row.row,
            'column': row.column,
            'source_blob_path': image_blob_path,
            'local_source_image_path': str(local_source_image_path) if local_source_image_path else '',
            'crop_output_path': crop_output_path,
            'crop_destination_mode': CROP_DESTINATION_MODE,
            'crop_width': right - left,
            'crop_height': bottom - top,
            'crop_left': left,
            'crop_top': top,
        })

manifest_df = pd.DataFrame(results)
summary_df = manifest_df.groupby('class_name').size().rename('crop_count').reset_index() if not manifest_df.empty else pd.DataFrame(columns=['class_name', 'crop_count'])

manifest_df.to_csv(LOCAL_MANIFEST_PATH, index=False)
summary_df.to_csv(LOCAL_COUNTS_PATH, index=False)

print({'crop_destination_mode': CROP_DESTINATION_MODE, 'written_crops': written_crops, 'downloaded_source_images': downloaded_source_images})
print(f'Skipped: {dict(skipped)}')
print({'missing_unique_images': len(missing_images), 'ambiguous_unique_images': len(ambiguous_images)})
if missing_images:
    print('First 10 missing image names:', sorted(missing_images)[:10])
if ambiguous_images:
    print('First 10 ambiguous image names:', sorted(ambiguous_images)[:10])
display(summary_df)

Extracting crops: 100%|██████████| 6140/6140 [47:17<00:00,  2.16it/s]  


{'crop_destination_mode': 'local', 'written_crops': 43906, 'downloaded_source_images': 4289}
Skipped: {'missing_source_image': 6672}
{'missing_unique_images': 1851, 'ambiguous_unique_images': 0}
First 10 missing image names: ['AGU-523_2022_A_23.JPG', 'AGU-532_2022_A_09.JPG', 'AGU-532_2022_A_20.JPG', 'AGU-541_2022_A_09.JPG', 'AGU-541_2022_A_26.JPG', 'AGU-541_2022_A_28.JPG', 'AGU-546_2022_A_07.JPG', 'AGU-546_2022_A_19.JPG', 'AGU-546_2022_A_30.JPG', 'AGU-548_2022_A_11.JPG']


,class_name,crop_count
0,CORAL,6492
1,CORAL_BL,3930
2,UNK,33484


In [14]:
if CROP_DESTINATION_MODE == 'gcs':
    manifest_output_path = build_blob_path(crops_prefix, '_manifests/crop_manifest.csv')
    counts_output_path = build_blob_path(crops_prefix, '_manifests/class_counts.csv')

    crops_bucket.blob(manifest_output_path).upload_from_filename(LOCAL_MANIFEST_PATH, content_type='text/csv')
    crops_bucket.blob(counts_output_path).upload_from_filename(LOCAL_COUNTS_PATH, content_type='text/csv')
else:
    manifest_output_path = LOCAL_MANIFEST_DIR / 'crop_manifest.csv'
    counts_output_path = LOCAL_MANIFEST_DIR / 'class_counts.csv'

    manifest_df.to_csv(manifest_output_path, index=False)
    summary_df.to_csv(counts_output_path, index=False)

print({
    'crop_destination_mode': CROP_DESTINATION_MODE,
    'manifest_output_path': str(manifest_output_path),
    'counts_output_path': str(counts_output_path),
})

{'crop_destination_mode': 'local', 'manifest_output_path': '/home/user/bleaching_classifier_training/crops/_manifests/crop_manifest.csv', 'counts_output_path': '/home/user/bleaching_classifier_training/crops/_manifests/class_counts.csv'}


In [15]:
manifest_df.head()

,annotation_id,image_name,label_code,class_name,row,column,source_blob_path,local_source_image_path,crop_output_path,crop_destination_mode,crop_width,crop_height,crop_left,crop_top
0,175,FFS-B009_2019_01.JPG,*UNK,UNK,204,976,PIFSC/ESD/ARP/pifsc-ai-data-repository/coralne...,/home/user/bleaching_classifier_training/sourc...,/home/user/bleaching_classifier_training/crops...,local,224,224,864,92
1,176,FFS-B009_2019_01.JPG,*UNK,UNK,637,1974,PIFSC/ESD/ARP/pifsc-ai-data-repository/coralne...,/home/user/bleaching_classifier_training/sourc...,/home/user/bleaching_classifier_training/crops...,local,224,224,1862,525
2,177,FFS-B009_2019_01.JPG,*UNK,UNK,882,2772,PIFSC/ESD/ARP/pifsc-ai-data-repository/coralne...,/home/user/bleaching_classifier_training/sourc...,/home/user/bleaching_classifier_training/crops...,local,224,224,2660,770
3,178,FFS-B009_2019_01.JPG,*UNK,UNK,206,3262,PIFSC/ESD/ARP/pifsc-ai-data-repository/coralne...,/home/user/bleaching_classifier_training/sourc...,/home/user/bleaching_classifier_training/crops...,local,224,224,3150,94
4,179,FFS-B009_2019_01.JPG,*UNK,UNK,209,4335,PIFSC/ESD/ARP/pifsc-ai-data-repository/coralne...,/home/user/bleaching_classifier_training/sourc...,/home/user/bleaching_classifier_training/crops...,local,224,224,3424,97


# Prepare Local Training Data

If `CROP_DESTINATION_MODE` is set to `gcs`, this section pulls the crops bucket down to the cloud workstation with `gsutil`. If `CROP_DESTINATION_MODE` is set to `local`, the crop extraction section already writes directly into the local training folder and this section becomes a lightweight verification step.

In [8]:
import subprocess

if CROP_DESTINATION_MODE == 'gcs':
    gsutil_command = [
        'gsutil',
        '-m',
        'rsync',
        '-r',
        CROPS_BUCKET_URI,
        str(LOCAL_CROPS_DIR),
    ]

    print('Running:', ' '.join(gsutil_command))
    subprocess.run(gsutil_command, check=True)
else:
    print('Skipping gsutil download because crops were written directly to the local training folder.')

print({
    'crop_destination_mode': CROP_DESTINATION_MODE,
    'local_training_root': str(LOCAL_TRAINING_ROOT),
    'local_source_images_dir': str(LOCAL_SOURCE_IMAGES_DIR),
    'local_crops_dir': str(LOCAL_CROPS_DIR),
    'local_manifest_dir': str(LOCAL_MANIFEST_DIR),
})

Skipping gsutil download because crops were written directly to the local training folder.
{'crop_destination_mode': 'local', 'local_training_root': '/home/user/bleaching_classifier_training', 'local_source_images_dir': '/home/user/bleaching_classifier_training/source_images', 'local_crops_dir': '/home/user/bleaching_classifier_training/crops', 'local_manifest_dir': '/home/user/bleaching_classifier_training/crops/_manifests'}


# Seeded Dataset Split

This section creates a deterministic train, validation, and test split from the downloaded crops. It also supports optional class balancing for the training split only, while leaving validation and test unchanged so evaluation still reflects the real class distribution.

In [17]:
import random
import shutil
import statistics
from collections import Counter
from pathlib import Path

RANDOM_SEED = 20260707
SPLIT_RATIOS = {'train': 0.7, 'val': 0.15, 'test': 0.15}
BALANCE_TRAINING_SET = True
BALANCE_MODE = 'hybrid'  # 'undersample', 'oversample', 'hybrid'
BALANCE_REFERENCE = 'median'  # 'min', 'max', 'median'

source_dir = LOCAL_CROPS_DIR
output_dir = LOCAL_TRAINING_ROOT / 'dataset_split'
split_manifest_path = LOCAL_TRAINING_ROOT / 'dataset_split_manifest.csv'

if abs(sum(SPLIT_RATIOS.values()) - 1.0) > 1e-9:
    raise ValueError(f'Split ratios must sum to 1.0, got {sum(SPLIT_RATIOS.values())}')

if BALANCE_MODE not in {'undersample', 'oversample', 'hybrid'}:
    raise ValueError(f'Unsupported BALANCE_MODE: {BALANCE_MODE}')
if BALANCE_REFERENCE not in {'min', 'max', 'median'}:
    raise ValueError(f'Unsupported BALANCE_REFERENCE: {BALANCE_REFERENCE}')

if output_dir.exists():
    shutil.rmtree(output_dir)

categories = sorted(
    path.name
    for path in source_dir.iterdir()
    if path.is_dir() and not path.name.startswith('_')
)

print({
    'categories': categories,
    'random_seed': RANDOM_SEED,
    'balance_training_set': BALANCE_TRAINING_SET,
    'balance_mode': BALANCE_MODE,
    'balance_reference': BALANCE_REFERENCE,
})

for split in SPLIT_RATIOS:
    for category in categories:
        split_category_dir = output_dir / split / category
        split_category_dir.mkdir(parents=True, exist_ok=True)

rng = random.Random(RANDOM_SEED)
split_rows = []
split_assignments = {category: {'train': [], 'val': [], 'test': []} for category in categories}

for category in categories:
    category_dir = source_dir / category
    images = sorted(path.name for path in category_dir.iterdir() if path.is_file())
    rng.shuffle(images)

    train_count = int(len(images) * SPLIT_RATIOS['train'])
    val_count = int(len(images) * SPLIT_RATIOS['val'])

    split_assignments[category]['train'] = images[:train_count]
    split_assignments[category]['val'] = images[train_count:train_count + val_count]
    split_assignments[category]['test'] = images[train_count + val_count:]

train_counts_before_balance = {
    category: len(split_assignments[category]['train'])
    for category in categories
}

def resolve_target_count(counts: dict[str, int], reference: str) -> int:
    values = list(counts.values())
    if reference == 'min':
        return min(values)
    if reference == 'max':
        return max(values)
    return int(statistics.median(values))


def balance_training_images(image_names: list[str], target_count: int, mode: str, rng: random.Random) -> list[str]:
    if not image_names:
        return []
    current_count = len(image_names)

    if mode == 'undersample':
        if current_count <= target_count:
            return list(image_names)
        return rng.sample(image_names, target_count)

    if mode == 'oversample':
        if current_count >= target_count:
            return list(image_names)
        extra = [rng.choice(image_names) for _ in range(target_count - current_count)]
        return list(image_names) + extra

    balanced = list(image_names)
    if current_count > target_count:
        balanced = rng.sample(balanced, target_count)
    elif current_count < target_count:
        balanced = balanced + [rng.choice(image_names) for _ in range(target_count - current_count)]
    return balanced

target_train_count = resolve_target_count(train_counts_before_balance, BALANCE_REFERENCE)
train_counts_after_balance = {}

for category in categories:
    if BALANCE_TRAINING_SET:
        balanced_train_images = balance_training_images(
            split_assignments[category]['train'],
            target_train_count,
            BALANCE_MODE,
            rng,
        )
    else:
        balanced_train_images = list(split_assignments[category]['train'])

    split_assignments[category]['train'] = balanced_train_images
    train_counts_after_balance[category] = len(balanced_train_images)

for category in categories:
    category_dir = source_dir / category

    for split_name in ['train', 'val', 'test']:
        selected_images = split_assignments[category][split_name]
        duplicate_counts = Counter(selected_images)
        written_counts = Counter()

        for image_name in selected_images:
            src = category_dir / image_name
            suffix = Path(image_name).suffix
            stem = Path(image_name).stem

            if split_name == 'train' and duplicate_counts[image_name] > 1:
                written_counts[image_name] += 1
                dest_name = f'{stem}__dup{written_counts[image_name]:03d}{suffix}'
            else:
                dest_name = image_name

            dest = output_dir / split_name / category / dest_name
            shutil.copy2(src, dest)

            split_rows.append({
                'split': split_name,
                'class_name': category,
                'image_name': image_name,
                'dest_image_name': dest_name,
                'source_path': str(src),
                'dest_path': str(dest),
                'random_seed': RANDOM_SEED,
                'training_balanced': BALANCE_TRAINING_SET and split_name == 'train',
                'balance_mode': BALANCE_MODE if BALANCE_TRAINING_SET and split_name == 'train' else '',
                'balance_reference': BALANCE_REFERENCE if BALANCE_TRAINING_SET and split_name == 'train' else '',
            })

split_manifest_df = pd.DataFrame(split_rows)
split_manifest_df.to_csv(split_manifest_path, index=False)

split_summary_df = split_manifest_df.groupby(['split', 'class_name']).size().rename('image_count').reset_index()
train_balance_summary_df = pd.DataFrame({
    'class_name': categories,
    'train_count_before_balance': [train_counts_before_balance[category] for category in categories],
    'train_count_after_balance': [train_counts_after_balance[category] for category in categories],
})

display(split_summary_df)
display(train_balance_summary_df)
print({
    'output_dir': str(output_dir),
    'split_manifest_path': str(split_manifest_path),
    'balance_training_set': BALANCE_TRAINING_SET,
    'balance_mode': BALANCE_MODE if BALANCE_TRAINING_SET else None,
    'balance_reference': BALANCE_REFERENCE if BALANCE_TRAINING_SET else None,
    'target_train_count': target_train_count if BALANCE_TRAINING_SET else None,
})

{'categories': ['CORAL', 'CORAL_BL', 'UNK'], 'random_seed': 20260707, 'balance_training_set': True, 'balance_mode': 'hybrid', 'balance_reference': 'median'}


,split,class_name,image_count
0,test,CORAL,975
1,test,CORAL_BL,590
2,test,UNK,5024
3,train,CORAL,4544
4,train,CORAL_BL,4544
5,train,UNK,4544
6,val,CORAL,973
7,val,CORAL_BL,589
8,val,UNK,5022


,class_name,train_count_before_balance,train_count_after_balance
0,CORAL,4544,4544
1,CORAL_BL,2751,4544
2,UNK,23438,4544


{'output_dir': '/home/user/bleaching_classifier_training/dataset_split', 'split_manifest_path': '/home/user/bleaching_classifier_training/dataset_split_manifest.csv', 'balance_training_set': True, 'balance_mode': 'hybrid', 'balance_reference': 'median', 'target_train_count': 4544}


# Train From The Seeded Split

This section trains the classifier against the deterministic split created above. If training-set balancing is enabled in the split cell, training uses that balanced train folder while validation still uses the original class distribution.

# Install Training Dependencies

For the training section on a T4 cloud workstation, install the CUDA-enabled PyTorch wheels before importing `ultralytics`. This avoids accidentally ending up with a CPU-only `torch` build.

In [20]:
import importlib
import subprocess
import sys

PYTORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu124'
TRAINING_PACKAGES = ['ultralytics', 'onnx']

def package_missing(module_name: str) -> bool:
    try:
        importlib.import_module(module_name)
        return False
    except ModuleNotFoundError:
        return True

python_version = (sys.version_info.major, sys.version_info.minor)
python_version_str = f'{python_version[0]}.{python_version[1]}'

# PyTorch wheels are not available for Python 3.14 at the time of writing.
if python_version >= (3, 14):
    raise RuntimeError(
        'PyTorch GPU wheels are unavailable for this kernel Python version ('
        + python_version_str
        + '). Use a Python 3.10-3.12 kernel for training.\n\n'
        'Cloud workstation fix:\n'
        '1) conda create -n coral-train python=3.12 -y\n'
        '2) conda activate coral-train\n'
        '3) conda install -n coral-train pip -y\n'
        '4) python -m pip install --upgrade pip\n'
        '5) python -m pip install --index-url https://download.pytorch.org/whl/cu124 torch torchvision torchaudio\n'
        '6) python -m pip install ultralytics onnx\n'
        '7) switch this notebook kernel to the coral-train environment'
    )

# Some conda images are created without pip; bootstrap it before package installs.
try:
    import pip  # noqa: F401
except ModuleNotFoundError:
    ensurepip_result = subprocess.run([sys.executable, '-m', 'ensurepip', '--upgrade'], check=False)
    if ensurepip_result.returncode != 0:
        raise RuntimeError(
            'pip is not available in this environment and ensurepip failed. '
            'Run: conda install -n coral-train pip -y, then rerun this cell.'
        )

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)

torch_needs_install = package_missing('torch')
ultralytics_needs_install = package_missing('ultralytics')
onnx_needs_install = package_missing('onnx')

if torch_needs_install:
    subprocess.run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--index-url',
        PYTORCH_INDEX_URL,
        'torch',
        'torchvision',
        'torchaudio',
    ], check=True)

missing_training_packages = []
if ultralytics_needs_install:
    missing_training_packages.append('ultralytics')
if onnx_needs_install:
    missing_training_packages.append('onnx')

if missing_training_packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing_training_packages], check=True)

import torch

print({
    'python_version': python_version_str,
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device_count': torch.cuda.device_count(),
    'cuda_device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})

if not torch.cuda.is_available():
    print('CUDA is not available inside torch. Training will fall back to CPU unless the PyTorch CUDA install is fixed.')

RuntimeError: PyTorch GPU wheels are unavailable for this kernel Python version (3.14). Use a Python 3.10-3.12 kernel for training.

Cloud workstation fix:
1) conda create -n coral-train python=3.12 -y
2) conda activate coral-train
3) conda install -n coral-train pip -y
4) python -m pip install --upgrade pip
5) python -m pip install --index-url https://download.pytorch.org/whl/cu124 torch torchvision torchaudio
6) python -m pip install ultralytics onnx
7) switch this notebook kernel to the coral-train environment

In [10]:
import random
import shutil
import statistics
from collections import Counter
from pathlib import Path

RANDOM_SEED = 20260707
BALANCE_TRAINING_SET = True
BALANCE_MODE = 'hybrid'  # 'undersample', 'oversample', 'hybrid'
BALANCE_REFERENCE = 'median'  # 'min', 'max', 'median'

source_dir = LOCAL_CROPS_DIR
output_dir = LOCAL_TRAINING_ROOT / 'dataset_split'
split_manifest_path = LOCAL_TRAINING_ROOT / 'dataset_split_manifest.csv'

In [1]:
import os
import shutil
import subprocess
import sys

print({'python_executable': sys.executable})
print({
    'CUDA_VISIBLE_DEVICES': os.environ.get('CUDA_VISIBLE_DEVICES'),
    'PATH_has_var_lib_nvidia_bin': '/var/lib/nvidia/bin' in os.environ.get('PATH', ''),
    'LD_LIBRARY_PATH': os.environ.get('LD_LIBRARY_PATH'),
})

nvidia_smi_path = shutil.which('nvidia-smi') or '/var/lib/nvidia/bin/nvidia-smi'
print({'nvidia_smi_path': nvidia_smi_path})

try:
    smi = subprocess.run([nvidia_smi_path, '--query-gpu=name,driver_version,memory.total,memory.used', '--format=csv,noheader'], capture_output=True, text=True, check=True)
    print('nvidia-smi:')
    print(smi.stdout.strip())
except Exception as error:
    print({'nvidia_smi_check_error': str(error)})

try:
    import torch
    print({
        'torch_version': torch.__version__,
        'torch_cuda_version': torch.version.cuda,
        'cuda_available': torch.cuda.is_available(),
        'cuda_device_count': torch.cuda.device_count(),
        'cuda_device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    })
except Exception as error:
    print({'torch_check_error': str(error)})

{'python_executable': '/home/conda/envs/coral-train/bin/python'}
{'CUDA_VISIBLE_DEVICES': None, 'PATH_has_var_lib_nvidia_bin': False, 'LD_LIBRARY_PATH': None}
{'nvidia_smi_path': '/var/lib/nvidia/bin/nvidia-smi'}
{'nvidia_smi_check_error': "Command '['/var/lib/nvidia/bin/nvidia-smi', '--query-gpu=name,driver_version,memory.total,memory.used', '--format=csv,noheader']' returned non-zero exit status 12."}
{'torch_version': '2.6.0+cu124', 'torch_cuda_version': '12.4', 'cuda_available': False, 'cuda_device_count': 0, 'cuda_device_name': None}


In [14]:
import os
import glob
import subprocess

os.environ["PATH"] = "/var/lib/nvidia/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/var/lib/nvidia/lib64:/usr/lib/x86_64-linux-gnu:" + os.environ.get("LD_LIBRARY_PATH", "")

print({"dev_nodes": glob.glob("/dev/nvidia*")})

res = subprocess.run(
    ["/var/lib/nvidia/bin/nvidia-smi"],
    capture_output=True,
    text=True
)
print({"nvidia_smi_returncode": res.returncode})
print("stdout:\n", res.stdout)
print("stderr:\n", res.stderr)

{'dev_nodes': ['/dev/nvidiactl', '/dev/nvidia0', '/dev/nvidia-uvm-tools', '/dev/nvidia-uvm', '/dev/nvidia-modeset', '/dev/nvidia-caps']}
{'nvidia_smi_returncode': 0}
stdout:
 Wed Jul  8 00:44:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB / 

In [11]:
from pathlib import Path

import torch
from ultralytics import YOLO

dataset_path = LOCAL_TRAINING_ROOT / 'dataset_split'
models_dir = LOCAL_TRAINING_ROOT / 'models'
training_logs_dir = LOCAL_TRAINING_ROOT / 'training_logs'
models_dir.mkdir(parents=True, exist_ok=True)
training_logs_dir.mkdir(parents=True, exist_ok=True)

required_splits = ['train', 'val']
missing_splits = [split_name for split_name in required_splits if not (dataset_path / split_name).exists()]
if missing_splits:
    raise FileNotFoundError(
        f'Missing dataset split folders: {missing_splits}. Run the seeded dataset split cell before training.'
    )

available_classes = sorted(
    path.name
    for path in (dataset_path / 'train').iterdir()
    if path.is_dir()
 )

print({
    'dataset_path': str(dataset_path),
    'available_classes': available_classes,
    'random_seed': RANDOM_SEED,
    'cuda_available': torch.cuda.is_available(),
})

model = YOLO('yolo11m-cls.pt')

results = model.train(
    data=str(dataset_path),
    device=0 if torch.cuda.is_available() else 'cpu',
    epochs=500,
    imgsz=224,
    batch=64,
    amp=torch.cuda.is_available(),
    optimizer='AdamW',
    lr0=5e-4,
    lrf=0.01,
    weight_decay=0.001,
    patience=35,
    seed=RANDOM_SEED,
    deterministic=True,
    project=str(training_logs_dir),
    name='yolo11m_cls_bleaching_seeded_split',
)

best_weights_path = Path(results.save_dir) / 'weights' / 'best.pt'
final_model_path = models_dir / 'yolo11m-cls-bleaching-seeded-split.pt'

if best_weights_path.exists():
    final_model_path.write_bytes(best_weights_path.read_bytes())
    print(f'Best model copied to: {final_model_path}')
else:
    print(f'Best weights not found at expected path: {best_weights_path}')

try:
    model.export(format='onnx')
    print('ONNX model exported successfully!')
except Exception as error:
    print(f'ONNX export failed: {error}')

metrics = model.val(data=str(dataset_path), device=0 if torch.cuda.is_available() else 'cpu')
print(metrics)

{'dataset_path': '/home/user/bleaching_classifier_training/dataset_split', 'available_classes': ['CORAL', 'CORAL_BL', 'UNK'], 'random_seed': 20260707, 'cuda_available': False}
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.6.0+cu124 CPU (Intel Xeon CPU @ 2.30GHz)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/user/bleaching_classifier_training/dataset_split, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_rati

KeyboardInterrupt: 

In [5]:
from pathlib import Path

root = Path.home() / "bleaching_classifier_training"
matches = list(root.rglob("*.onnx"))
print([str(p) for p in matches])

['/home/user/bleaching_classifier_training/training_logs/yolo11m_cls_bleaching_seeded_split-3/weights/best.onnx']


In [3]:
import onnxruntime as ort

onnx_path = "/home/user/bleaching_classifier_training/training_logs/yolo11m_cls_bleaching_seeded_split-3/weights/best.onnx"
session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

print("inputs:", [x.name for x in session.get_inputs()])
print("outputs:", [x.name for x in session.get_outputs()])
print("output shape:", session.get_outputs()[0].shape)

inputs: ['images']
outputs: ['output0']
output shape: [1, 3]


In [2]:
from pathlib import Path
import numpy as np
from PIL import Image
import onnxruntime as ort

onnx_path = "/home/user/bleaching_classifier_training/training_logs/yolo11m_cls_bleaching_seeded_split-3/weights/best.onnx"
image_path = "/home/user//bleaching_classifier_training/dataset_split/test/CORAL_BL/KawaihaeShallow2015IMG_5205__ann002516__r810__c2686.jpg"

class_map = {
    0: "CORAL",
    1: "CORAL_BL",
    2: "UNK",
}

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name

img = Image.open(image_path).convert("RGB").resize((224, 224))
arr = np.asarray(img).astype(np.float32) / 255.0
arr = np.transpose(arr, (2, 0, 1))
arr = np.expand_dims(arr, axis=0)

logits = session.run(None, {input_name: arr})[0]
pred_idx = int(np.argmax(logits, axis=1)[0])

print("pred_idx:", pred_idx)
print("pred_class:", class_map[pred_idx])

pred_idx: 0
pred_class: CORAL


In [4]:
import onnx
import ast
import json

# Load the ONNX model
model_path = "/home/user/bleaching_classifier_training/training_logs/yolo11m_cls_bleaching_seeded_split-3/weights/best.onnx"
model = onnx.load(model_path)

# Convert metadata properties into a standard Python dictionary
metadata = {prop.key: prop.value for prop in model.metadata_props}

# Print all available metadata keys to look for label storage
print("Available metadata keys:", list(metadata.keys()))

# 1. Check for 'names' key (Standard for YOLO models)
if 'names' in metadata:
    class_list = ast.literal_eval(metadata['names'])
    print("\nClass List Order:")
    print(class_list)

# 2. Check for 'class_names' key (Standard for other common pipelines)
elif 'class_names' in metadata:
    class_list = json.loads(metadata['class_names'])
    print("\nClass List Order:")
    print(class_list)

Available metadata keys: ['description', 'author', 'date', 'version', 'license', 'docs', 'stride', 'task', 'head', 'batch', 'imgsz', 'names', 'args', 'channels', 'end2end']

Class List Order:
{0: 'CORAL', 1: 'CORAL_BL', 2: 'UNK'}


In [5]:
import onnx

# Load the ONNX model
model_path = "/home/user/bleaching_classifier_training/training_logs/yolo11m_cls_bleaching_seeded_split-3/weights/best.onnx"
model = onnx.load(model_path)

class_list = None

# Iterate through graph nodes to find ZipMap or LabelEncoder
for node in model.graph.node:
    if node.op_type in ["ZipMap", "LabelEncoder"]:
        for attr in node.attribute:
            # Look for attributes defining string or integer labels
            if attr.name in ["classlabels_string", "classlabels_int64"]:
                if attr.strings:
                    class_list = [s.decode('utf-8') for s in attr.strings]
                elif attr.ints:
                    class_list = list(attr.ints)
                break

if class_list is not None:
    print("Class List Order found in graph node:")
    for idx, label in enumerate(class_list):
        print(f"{idx}: {label}")
else:
    print("No ZipMap or LabelEncoder nodes containing class labels were found.")

No ZipMap or LabelEncoder nodes containing class labels were found.
